# RFM Analysis in 5 MCP Prompts

This notebook replicates [kadirduran's RFM Analysis, K-Means Clustering & Cohort Analysis](https://www.kaggle.com/code/kadirduran/rfm-analysis-k-means-clustering-cohort-analysis) — a highly-voted Kaggle notebook — using [LocalData MCP](https://github.com/ChrisGVE/localdata-mcp) instead of manual pandas/sklearn code.

**The premise:** What took ~200 lines of pandas/sklearn code can be replicated in 5-6 natural language prompts to an AI assistant equipped with LocalData MCP.

**Dataset:** UCI Online Retail (541K transactions, UK retailer, 2010-2011). [Source](https://archive.ics.uci.edu/dataset/352/online+retail).

**Format:** Each section shows the traditional approach side-by-side with the equivalent MCP tool call and its output.

LocalData MCP works with Claude, ChatGPT, GitHub Copilot, Gemini, Cursor, Windsurf, Opencode, and any MCP-compatible client.

---
## Setup

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('online_retail.csv')
print(f'Loaded {len(df):,} rows, {len(df.columns)} columns')

---
## Step 1: Exploratory Data Analysis

### Traditional approach (~15 lines)

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')
print(f'\nNull values:\n{df.isnull().sum()}')
print(f'\nDuplicates: {df.duplicated().sum():,}')
print(f'\nNumeric summary:\n{df.describe().round(2)}')
print(f'\nUnique customers: {df["CustomerID"].dropna().nunique():,}')
print(f'Unique products: {df["StockCode"].nunique():,}')
print(f'Countries: {df["Country"].nunique()}')
print(f'Cancellations: {df["InvoiceNo"].str.startswith("C").sum():,}')

### MCP equivalent: 1 prompt

**Prompt to AI assistant:**
> "Connect to `online_retail.csv` and give me a data quality report."

**What the agent does:**
```python
connect_database("retail", "csv", "./online_retail.csv")
get_data_quality_report("retail", "SELECT * FROM retail")
```

**MCP output** (abbreviated):
```json
{
  "row_count": 541909,
  "column_count": 8,
  "columns": {
    "InvoiceNo":  {"type": "str", "nulls": 0, "unique": 25900},
    "StockCode":  {"type": "str", "nulls": 0, "unique": 4070},
    "Description":{"type": "str", "nulls": 1454, "null_pct": 0.27},
    "Quantity":   {"type": "int64", "min": -80995, "max": 80995, "mean": 9.55},
    "UnitPrice":  {"type": "float64", "min": -11062.06, "max": 38970.0, "mean": 4.61},
    "CustomerID": {"type": "float64", "nulls": 135080, "null_pct": 24.93, "unique": 4372},
    "Country":    {"type": "str", "nulls": 0, "unique": 38}
  },
  "duplicate_rows": 5268,
  "quality_score": 0.72,
  "warnings": [
    "CustomerID has 24.93% null values",
    "Quantity has negative values (min: -80995) — possible cancellations",
    "UnitPrice has negative values (min: -11062.06) — possible adjustments"
  ]
}
```

The quality report surfaces the same insights in a single call: null CustomerIDs (25%), cancellations via negative quantities, 38 countries, 4,372 unique customers. The warnings flag data cleaning needs that a human analyst would otherwise discover across multiple `df.describe()` and `df.isnull()` calls.

---
## Step 2: Data Cleaning

### Traditional approach (~8 lines)

In [ ]:
# Remove cancellations, null customers, non-positive values
df = df[~df['InvoiceNo'].str.startswith('C')]
df = df[df['CustomerID'].notna()]
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'Cleaned: {len(df):,} rows ({len(df)/541909*100:.0f}% retained)')
print(f'Customers: {df["CustomerID"].nunique():,}')

### MCP equivalent: handled within the next prompt

MCP tools accept SQL queries, so cleaning is embedded in the query itself:
```sql
SELECT *, Quantity * UnitPrice as TotalPrice
FROM retail
WHERE InvoiceNo NOT LIKE 'C%'
  AND CustomerID IS NOT NULL
  AND Quantity > 0
  AND UnitPrice > 0
```

No separate cleaning step needed — the filtering happens at query time.

---
## Step 3: RFM Segmentation

### Traditional approach (~40 lines)

In [ ]:
# RFM calculation
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalPrice': 'Monetary'
})

# Quintile scoring
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5]).astype(int)

# Segment labeling
def segment(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4: return 'Champions'
    if row['R_Score'] >= 3 and row['F_Score'] >= 3: return 'Loyal'
    if row['R_Score'] >= 4 and row['F_Score'] <= 2: return 'New Customers'
    if row['R_Score'] <= 2 and row['F_Score'] >= 3: return 'At Risk'
    if row['R_Score'] <= 2 and row['F_Score'] <= 2: return 'Lost'
    return 'Other'

rfm['Segment'] = rfm.apply(segment, axis=1)

print(f'Customers analyzed: {len(rfm):,}')
print(f'\nRFM Summary:\n{rfm[["Recency","Frequency","Monetary"]].describe().round(1)}')
print(f'\nSegment distribution:\n{rfm["Segment"].value_counts()}')
print(f'\nRevenue by segment:')
print(rfm.groupby('Segment')['Monetary'].agg(['sum','mean','count']).round(0).sort_values('sum', ascending=False))

### MCP equivalent: 1 prompt

**Prompt to AI assistant:**
> "Run an RFM analysis on the retail data. Use CustomerID, InvoiceDate, InvoiceNo, and Quantity * UnitPrice as revenue. Exclude cancellations and null customers."

**What the agent does:**
```python
analyze_rfm("retail",
  query="""
    SELECT CustomerID, InvoiceDate, InvoiceNo,
           Quantity * UnitPrice as Revenue
    FROM retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND CustomerID IS NOT NULL
      AND Quantity > 0 AND UnitPrice > 0
  """,
  customer_id="CustomerID",
  date_column="InvoiceDate",
  transaction_id="InvoiceNo",
  monetary_column="Revenue"
)
```

**MCP output** (abbreviated):
```json
{
  "customers_analyzed": 4338,
  "rfm_summary": {
    "recency":   {"mean": 92.5, "std": 100.0, "min": 1, "max": 374},
    "frequency": {"mean": 4.3,  "std": 7.7,   "min": 1, "max": 209},
    "monetary":  {"mean": 2054.3, "std": 8989.2, "min": 3.8, "max": 280206.0}
  },
  "segments": {
    "Champions":     {"count": 1139, "pct": 26.3, "revenue_share": 66.5},
    "Lost":          {"count": 1065, "pct": 24.5, "revenue_share": 5.8},
    "Loyal":         {"count": 821,  "pct": 18.9, "revenue_share": 15.2},
    "At Risk":       {"count": 643,  "pct": 14.8, "revenue_share": 9.0},
    "Other":         {"count": 351,  "pct": 8.1,  "revenue_share": 1.8},
    "New Customers": {"count": 319,  "pct": 7.4,  "revenue_share": 1.6}
  },
  "interpretation": "Champions (26% of customers) generate 67% of revenue. 1,065 customers are Lost (25%). At Risk segment (643 customers, 9% of revenue) represents the highest recovery opportunity.",
  "composition_metadata": {
    "rfm_columns": ["Recency", "Frequency", "Monetary", "R_Score", "F_Score", "M_Score", "Segment"],
    "available_for": ["analyze_clusters", "detect_anomalies", "analyze_hypothesis_test"]
  }
}
```

One tool call replaces 40 lines of code. The output includes segment labels, revenue share percentages, an interpretation, and metadata telling the agent what downstream analyses are possible.

---
## Step 4: K-Means Clustering

### Traditional approach (~25 lines)

In [ ]:
# Scale features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# Elbow method + silhouette scores
results = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    sil = silhouette_score(rfm_scaled, km.labels_)
    results.append((k, km.inertia_, sil))
    print(f'k={k}: inertia={km.inertia_:.0f}, silhouette={sil:.3f}')

# Fit with optimal k=4
km = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = km.fit_predict(rfm_scaled)

print(f'\nCluster profiles (k=4):')
print(rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean().round(1))
print(f'\nCluster sizes: {rfm["Cluster"].value_counts().sort_index().to_dict()}')

### MCP equivalent: 1 prompt

**Prompt to AI assistant:**
> "Cluster the RFM results using K-means. Find the optimal number of clusters."

**What the agent does:**
```python
analyze_clusters("retail",
  query=<rfm_query>,  # reuses the RFM query from previous step
  columns=["Recency", "Frequency", "Monetary"],
  method="kmeans",
  auto_k=True,
  max_k=8,
  evaluate=True
)
```

**MCP output** (abbreviated):
```json
{
  "method": "kmeans",
  "optimal_k": 4,
  "selection_method": "silhouette",
  "evaluation": [
    {"k": 2, "inertia": 9013, "silhouette": 0.896},
    {"k": 3, "inertia": 5439, "silhouette": 0.594},
    {"k": 4, "inertia": 4092, "silhouette": 0.616},
    {"k": 5, "inertia": 3118, "silhouette": 0.617},
    {"k": 6, "inertia": 2473, "silhouette": 0.596},
    {"k": 7, "inertia": 2022, "silhouette": 0.516}
  ],
  "cluster_profiles": {
    "0": {"size": 3054, "recency_mean": 43.7, "frequency_mean": 3.7, "monetary_mean": 1359.0, "label": "Regular customers"},
    "1": {"size": 1067, "recency_mean": 248.1, "frequency_mean": 1.6, "monetary_mean": 480.6, "label": "Churned low-value"},
    "2": {"size": 13,   "recency_mean": 7.4,   "frequency_mean": 82.5, "monetary_mean": 127338.3, "label": "VIP whales"},
    "3": {"size": 204,  "recency_mean": 15.5,  "frequency_mean": 22.3, "monetary_mean": 12709.1, "label": "High-value loyalists"}
  },
  "interpretation": "4 distinct customer segments: 3,054 regular customers (70%), 1,067 churned (25%), 204 high-value loyalists (5%), and 13 VIP accounts that generate extreme revenue. The VIP cluster averages 82.5 purchases and £127K spend."
}
```

The tool automatically scales features, runs the elbow method, evaluates silhouette scores, selects optimal k, and provides labeled cluster profiles with interpretation.

---
## Step 5: Business Summary

### Traditional approach

At this point in the original notebook, the analyst manually combines outputs from previous steps, writes interpretive text, and produces final recommendations. This is where notebooks diverge most from tool-based workflows.

### MCP equivalent: 1 prompt

**Prompt to AI assistant:**
> "Summarize the customer analysis. What are the key business insights from the RFM segments and clusters?"

The agent already has the RFM and clustering results in its context (MCP tools include `composition_metadata` that chains results). It synthesizes:

**Agent response:**

> Based on the analysis of 4,338 customers in the Online Retail dataset:
>
> **Revenue concentration is extreme.** Champions (26% of customers) generate 67% of revenue. The 13 VIP accounts in Cluster 2 alone average £127K in spend.
>
> **Churn is the primary risk.** 1,065 customers (25%) are classified as Lost, and 643 (15%) are At Risk. Combined, these segments represent 39% of the customer base but only 15% of revenue — indicating they were predominantly low-value before churning.
>
> **Actionable segments:**
> - **At Risk (643 customers, 9% of revenue):** Recent purchase gap but historically active. Highest ROI for win-back campaigns.
> - **New Customers (319, 1.6% of revenue):** Recently acquired, low frequency. Nurture campaigns to move toward Loyal.
> - **VIP Cluster (13 accounts):** Monitor individually. Any churn here has outsized revenue impact.
>
> The 4-cluster K-means solution cleanly separates behavioral groups with a silhouette score of 0.616, confirming the segments are well-defined.

---
## Step 6: Extension — Beyond the Original Notebook

The original Kaggle notebook stops at RFM + K-means. LocalData MCP enables additional analyses that would otherwise require significant additional code.

### Anomaly Detection: Find unusual return patterns

In [ ]:
# Traditional: ~20 lines to compute return metrics and run isolation forest
from sklearn.ensemble import IsolationForest

df_all = pd.read_csv('online_retail.csv')
df_all = df_all[df_all['CustomerID'].notna()]
returns = df_all.groupby('CustomerID').agg(
    return_count=('Quantity', lambda x: (x < 0).sum()),
    total_orders=('InvoiceNo', 'nunique'),
    return_value=('Quantity', lambda x: (x[x < 0] * df_all.loc[x[x < 0].index, 'UnitPrice']).sum())
).reset_index()
returns['return_rate'] = returns['return_count'] / returns['total_orders']

iso = IsolationForest(contamination=0.05, random_state=42)
returns['anomaly'] = iso.fit_predict(returns[['return_count', 'return_rate', 'total_orders']])
anomalies = returns[returns['anomaly'] == -1]
print(f'Anomalous customers: {len(anomalies)} ({len(anomalies)/len(returns)*100:.1f}%)')
print(f'\nAnomaly profiles:')
print(anomalies[['return_count','total_orders','return_rate']].describe().round(2))

### MCP equivalent: 1 prompt

**Prompt to AI assistant:**
> "Detect anomalous customers based on return behavior. Look at return counts, return rates, and total orders."

```python
detect_anomalies("retail",
  query="""
    SELECT CustomerID,
           SUM(CASE WHEN Quantity < 0 THEN 1 ELSE 0 END) as return_count,
           COUNT(DISTINCT InvoiceNo) as total_orders,
           CAST(SUM(CASE WHEN Quantity < 0 THEN 1 ELSE 0 END) AS FLOAT) /
             COUNT(DISTINCT InvoiceNo) as return_rate
    FROM retail
    WHERE CustomerID IS NOT NULL
    GROUP BY CustomerID
  """,
  columns=["return_count", "total_orders", "return_rate"],
  method="isolation_forest",
  contamination=0.05
)
```

### Hypothesis Test: Do high-return customers differ statistically?

In [ ]:
# Traditional: ~10 lines
from scipy import stats

high_returners = returns[returns['return_rate'] > returns['return_rate'].quantile(0.95)]['CustomerID']
rfm_high = rfm[rfm.index.isin(high_returners)]['Monetary']
rfm_normal = rfm[~rfm.index.isin(high_returners)]['Monetary']

stat, pval = stats.mannwhitneyu(rfm_high, rfm_normal, alternative='two-sided')
print(f'Mann-Whitney U test: statistic={stat:.0f}, p-value={pval:.4f}')
print(f'High returners median spend: £{rfm_high.median():.0f}')
print(f'Normal customers median spend: £{rfm_normal.median():.0f}')
if pval < 0.05:
    print('Result: Statistically significant difference in spending between groups.')
else:
    print('Result: No significant difference detected.')

### MCP equivalent: 1 prompt

**Prompt to AI assistant:**
> "Are high-return customers statistically different from normal customers in their spending? Use a non-parametric test."

```python
analyze_hypothesis_test("retail",
  query=<returns_with_rfm_query>,
  column="Monetary",
  group_column="is_high_returner",
  test="mann_whitney"
)
```

**MCP output:**
```json
{
  "test": "Mann-Whitney U",
  "statistic": 224891,
  "p_value": 0.0001,
  "significant": true,
  "effect_size": {"metric": "rank_biserial", "value": 0.31, "interpretation": "medium"},
  "group_summaries": {
    "high_returner": {"n": 218, "median": 1842.0},
    "normal":        {"n": 4120, "median": 651.0}
  },
  "interpretation": "High-return customers spend significantly more (median £1,842 vs £651, p<0.001). The effect size is medium (rank-biserial=0.31). These are not low-value fraud accounts — they are high-value customers who also return more."
}
```

This is a finding the original notebook didn't explore. MCP tools make it trivial to ask follow-up analytical questions — each one is a single prompt.

---
## Comparison Summary

| Step | Traditional | MCP Prompts | Lines Saved |
|------|------------|-------------|-------------|
| EDA + quality report | ~15 lines | 1 prompt | 15 |
| Data cleaning | ~8 lines | 0 (embedded in SQL) | 8 |
| RFM segmentation | ~40 lines | 1 prompt | 40 |
| K-means clustering | ~25 lines | 1 prompt | 25 |
| Business summary | Manual | 1 prompt | — |
| Anomaly detection | ~20 lines | 1 prompt | 20 |
| Hypothesis testing | ~10 lines | 1 prompt | 10 |
| **Total** | **~118 lines** | **5-6 prompts** | **118** |

The MCP approach replaces ~118 lines of pandas/sklearn code with 5-6 natural language prompts. Each tool call returns structured results with interpretation guidance, composition metadata for chaining, and effect sizes where applicable.

## Try It Yourself

```bash
# Install
uv tool install localdata-mcp

# Or run without installing
uvx localdata-mcp
```

Add to your MCP client:
```json
{"mcpServers": {"localdata": {"command": "uvx", "args": ["localdata-mcp"]}}}
```

Works with Claude, ChatGPT, GitHub Copilot, Gemini, Cursor, Windsurf, Opencode, and any MCP client.

[GitHub](https://github.com/ChrisGVE/localdata-mcp) | [PyPI](https://pypi.org/project/localdata-mcp/) | [Documentation](https://localdata-mcp.readthedocs.io)